In [2]:
import gzip
import pandas as pd
import os

def load_oas_file(filepath, isotype_label):
    df = pd.read_csv(filepath, skiprows=1, compression='gzip')
    df = df[['sequence_alignment_aa', 'v_call', 'productive']].copy()
    df = df[df['productive'] == 'T'].drop(columns='productive')
    df = df.dropna()
    df['v_family'] = df['v_call'].str.extract(r'(IGHV\d+)')
    df['isotype'] = isotype_label
    return df

base = os.path.expanduser("~/igbert-embedding-analysis/data/raw/")

df_ighg = load_oas_file(base + "SRR13082916_1_Heavy_IGHG.csv.gz", "IGHG")
df_igha = load_oas_file(base + "SRR13082911_1_Heavy_IGHA.csv.gz", "IGHA")
df_ighm = load_oas_file(base + "SRR13082916_1_Heavy_IGHM.csv.gz", "IGHM")

print(f"IGHG: {len(df_ighg)} sequences")
print(f"IGHA: {len(df_igha)} sequences")
print(f"IGHM: {len(df_ighm)} sequences")

IGHG: 17052 sequences
IGHA: 21892 sequences
IGHM: 8541 sequences


In [3]:
df_all = pd.concat([df_ighg, df_igha, df_ighm], ignore_index=True)

families_to_keep = ['IGHV1', 'IGHV3', 'IGHV4']
df_all = df_all[df_all['v_family'].isin(families_to_keep)]

df_balanced = (
    df_all.groupby(['v_family', 'isotype'])
    .sample(n=500, random_state=42)
    .reset_index(drop=True)
)

print(f"Σύνολο sequences: {len(df_balanced)}")
print(f"\nΚατανομή:")
print(df_balanced.groupby(['v_family', 'isotype']).size().unstack())

Σύνολο sequences: 4500

Κατανομή:
isotype   IGHA  IGHG  IGHM
v_family                  
IGHV1      500   500   500
IGHV3      500   500   500
IGHV4      500   500   500


In [4]:
def load_oas_file_extended(filepath, isotype_label):
    df = pd.read_csv(filepath, skiprows=1, compression='gzip')
    
    # Κρατάμε τώρα περισσότερες στήλες
    df = df[['sequence_alignment_aa', 'v_call', 'j_call', 
             'junction_aa_length', 'v_identity', 'productive']].copy()
    
    df = df[df['productive'] == 'T'].drop(columns='productive')
    df = df.dropna()
    
    df['v_family'] = df['v_call'].str.extract(r'(IGHV\d+)')
    df['j_family'] = df['j_call'].str.extract(r'(IGHJ\d+)')
    df['isotype'] = isotype_label
    
    return df

base = os.path.expanduser("~/igbert-embedding-analysis/data/raw/")

df_ighg = load_oas_file_extended(base + "SRR13082916_1_Heavy_IGHG.csv.gz", "IGHG")
df_igha = load_oas_file_extended(base + "SRR13082911_1_Heavy_IGHA.csv.gz", "IGHA")
df_ighm = load_oas_file_extended(base + "SRR13082916_1_Heavy_IGHM.csv.gz", "IGHM")

df_all = pd.concat([df_ighg, df_igha, df_ighm], ignore_index=True)
families_to_keep = ['IGHV1', 'IGHV3', 'IGHV4']
df_all = df_all[df_all['v_family'].isin(families_to_keep)]

df_balanced = (
    df_all.groupby(['v_family', 'isotype'])
    .sample(n=500, random_state=42)
    .reset_index(drop=True)
)

# Αποθήκευση του ενημερωμένου dataset
output_path = os.path.expanduser("~/igbert-embedding-analysis/data/processed/sequences_balanced.csv")
df_balanced.to_csv(output_path, index=False)

print(f"Σύνολο sequences: {len(df_balanced)}")
print(f"\nΣτήλες: {df_balanced.columns.tolist()}")
print(f"\nΔείγμα:")
print(df_balanced.head(3))

Σύνολο sequences: 4500

Στήλες: ['sequence_alignment_aa', 'v_call', 'j_call', 'junction_aa_length', 'v_identity', 'v_family', 'j_family', 'isotype']

Δείγμα:
                               sequence_alignment_aa       v_call    j_call  \
0  ASVKVSCKVSGYTLTELSLHWVRQAPGKGLEWMGGFDPEDGKTIYA...  IGHV1-24*01  IGHJ4*02   
1  SVKVSCKASGGTFSSYAISWVRQAPGQGLEWMGGIIPIFGTANYAQ...  IGHV1-69*13  IGHJ6*02   
2  ASVKVSCKASGYTFTGYYMHWVRQAPGQGLEWMGWINPNSGGTNYA...   IGHV1-2*02  IGHJ6*02   

   junction_aa_length  v_identity v_family j_family isotype  
0                21.0      95.618    IGHV1    IGHJ4    IGHA  
1                22.0     100.000    IGHV1    IGHJ6    IGHA  
2                19.0     100.000    IGHV1    IGHJ6    IGHA  
